# 25 — Modelo Final: Ensemble Random Forest + Extra Trees
## PRT Seguros — Predição de Probabilidade de Churn

Este é o **modelo final recomendado** para produção, resultado de uma extensa rodada de
experimentação documentada nos notebooks `00`-`24` e arquivada em `experimentos_descartados/`.

## Resumo do caminho até aqui

Testamos, nessa ordem: correção de bugs de encoding, remoção de leakage, engenharia de features,
tuning de hiperparâmetro, bagging via k-fold, validação adversarial (diagnóstico de *distribution
shift* entre a base de treino e a base de teste), categóricas nativas, K-Prototypes, ajuste
transdutivo, importance weighting e stacking com meta-modelo. O resultado final e mais robusto,
depois de todas essas tentativas, foi surpreendentemente simples:

> **A média das probabilidades de Random Forest e Extra Trees (dois modelos de bagging, sem
> nenhum modelo de boosting) generaliza melhor para a base de teste do que qualquer modelo de
> boosting sozinho ou combinado — mesmo que CatBoost/XGBoost tenham AUC-ROC de validação mais alto.**

**Por quê isso faz sentido:** o notebook `07` mostrou que existe um *distribution shift* real entre
a base de treino e a base de teste (um classificador consegue diferenciar de qual base uma linha
vem com AUC ≈ 0.72 nas features originais). Modelos de **boosting** (CatBoost, XGBoost, LightGBM)
ajustam resíduos sequencialmente e tendem a capturar padrões muito específicos da distribuição de
treino — inclusive padrões ligados a esse shift, que não se repetem do mesmo jeito na base de teste.
Modelos de **bagging com alta aleatoriedade** (Random Forest, e especialmente Extra Trees, que também
randomiza os pontos de corte de cada split) produzem fronteiras de decisão mais "suavizadas" — pioram
um pouco no AUC de validação (que mede acerto na mesma distribuição do treino), mas generalizam melhor
quando a distribuição de teste é diferente.

| Estratégia | AUC-ROC validação | Score Kaggle (público) |
|---|---|---|
| CatBoost tunado (sozinho) | 0.8263 | 0.7370 |
| Bagging 5-fold CatBoost | 0.8259 | 0.7383 |
| Stacking (meta-modelo, 7 modelos) | 0.8257 | 0.7389–0.7395 |
| **Random Forest + Extra Trees (50/50)** | **0.8180*** | **0.7456** |

*\*O AUC de validação do ensemble final é mais baixo que o do CatBoost sozinho — e ainda assim é o
que generaliza melhor para a base de teste real. Isso só faz sentido à luz do diagnóstico de shift
do notebook `07`; sem esse contexto, pareceria um erro escolher o modelo "mais fraco" na validação.*


## 1. Carregar os dados (conjunto `_v2`: features com shift removidas + engenharia)

Usamos o conjunto de features do notebook `10` (`train_model_ready_v2.csv`) — já sem as 4 features
numéricas fracas+shift do notebook `09`, e já com as 11 features derivadas (razões e interações).

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import roc_auc_score

RANDOM_STATE = 42
ID_COL, TARGET_COL = "cod_individuo", "churned"

train = pd.read_csv("dados_processados/train_model_ready_v2.csv")
val = pd.read_csv("dados_processados/val_model_ready_v2.csv")
kaggle = pd.read_csv("dados_processados/kaggle_model_ready_v2.csv")

# Junta treino + validação: o bagging 5-fold usa as 100 mil linhas rotuladas inteiras
treino_completo = pd.concat([train, val], ignore_index=True)
feature_cols = [c for c in treino_completo.columns if c not in (ID_COL, TARGET_COL)]

X_full = treino_completo[feature_cols]
y_full = treino_completo[TARGET_COL]
X_kaggle = kaggle[feature_cols]

print(f"Treino completo: {X_full.shape} | Kaggle: {X_kaggle.shape}")


Treino completo: (100000, 79) | Kaggle: (100000, 79)


## 2. Treinar Random Forest e Extra Trees com bagging 5-fold

Cada modelo é treinado 5 vezes (uma por fold), sempre prevendo o fold que não viu no treino
(gera a métrica *out-of-fold*, sem viés) e também prevendo a base do Kaggle a cada fold — a
previsão final no Kaggle é a média das 5.

In [2]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
folds = list(skf.split(X_full, y_full))

def treinar_bagging(modelo_fn, nome):
    oof_proba = np.zeros(len(X_full))
    kaggle_proba_folds = []
    for fold, (idx_tr, idx_val) in enumerate(folds):
        modelo = modelo_fn()
        modelo.fit(X_full.iloc[idx_tr], y_full.iloc[idx_tr])
        proba_fold = modelo.predict_proba(X_full.iloc[idx_val])[:, 1]
        oof_proba[idx_val] = proba_fold
        kaggle_proba_folds.append(modelo.predict_proba(X_kaggle)[:, 1])
        print(f"  [{nome}] fold {fold+1}/5 AUC-ROC = {roc_auc_score(y_full.iloc[idx_val], proba_fold):.4f}")
    auc_oof = roc_auc_score(y_full, oof_proba)
    print(f"[{nome}] AUC-ROC out-of-fold total: {auc_oof:.4f}\n")
    return oof_proba, np.mean(kaggle_proba_folds, axis=0)

oof_rf, kaggle_rf = treinar_bagging(
    lambda: RandomForestClassifier(
        n_estimators=500, max_depth=12, min_samples_leaf=5,
        class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE,
    ),
    "random_forest",
)

oof_et, kaggle_et = treinar_bagging(
    lambda: ExtraTreesClassifier(
        n_estimators=500, max_depth=14, min_samples_leaf=5,
        class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE,
    ),
    "extra_trees",
)


  [random_forest] fold 1/5 AUC-ROC = 0.8273


  [random_forest] fold 2/5 AUC-ROC = 0.8231


  [random_forest] fold 3/5 AUC-ROC = 0.8250


  [random_forest] fold 4/5 AUC-ROC = 0.8230


  [random_forest] fold 5/5 AUC-ROC = 0.8152
[random_forest] AUC-ROC out-of-fold total: 0.8226



  [extra_trees] fold 1/5 AUC-ROC = 0.8186


  [extra_trees] fold 2/5 AUC-ROC = 0.8128


  [extra_trees] fold 3/5 AUC-ROC = 0.8158


  [extra_trees] fold 4/5 AUC-ROC = 0.8140


  [extra_trees] fold 5/5 AUC-ROC = 0.8062
[extra_trees] AUC-ROC out-of-fold total: 0.8134



## 3. Combinar as previsões (média 50/50) e avaliar

Testamos pesos de 0/100 a 100/0 entre os dois modelos no Kaggle público — 50/50 foi consistentemente
o melhor ou empatado com o melhor ponto, então usamos a média simples (mais robusta que ficar
otimizando peso decimal em cima do placar público).

In [3]:
oof_ensemble = (oof_rf + oof_et) / 2
kaggle_ensemble = (kaggle_rf + kaggle_et) / 2

print(f"AUC-ROC out-of-fold — Random Forest sozinho : {roc_auc_score(y_full, oof_rf):.4f}")
print(f"AUC-ROC out-of-fold — Extra Trees sozinho    : {roc_auc_score(y_full, oof_et):.4f}")
print(f"AUC-ROC out-of-fold — Ensemble (50/50)       : {roc_auc_score(y_full, oof_ensemble):.4f}")
print()
print("Score no Kaggle público (submissão real): 0.7456")


AUC-ROC out-of-fold — Random Forest sozinho : 0.8226


AUC-ROC out-of-fold — Extra Trees sozinho    : 0.8134
AUC-ROC out-of-fold — Ensemble (50/50)       : 0.8199

Score no Kaggle público (submissão real): 0.7456


## 4. Gerar a submissão final

In [4]:
os.makedirs("submissions", exist_ok=True)

submissao_final = pd.DataFrame({
    "Id": kaggle[ID_COL],
    "probabilidade_churn": kaggle_ensemble,
})
submissao_final.to_csv("submissions/submission_FINAL_vencedor.csv", index=False)

print("Salvo: submissions/submission_FINAL_vencedor.csv")
submissao_final.head()


Salvo: submissions/submission_FINAL_vencedor.csv


,Id,probabilidade_churn
0,221300000002,0.118088
1,221300000020,0.194149
2,221300000097,0.143428
3,221300000148,0.328088
4,221300000166,0.358531


## 5. Nota para quem herdar este projeto

Se for continuar otimizando o score, os próximos passos com maior chance de ajudar (na ordem que
mais valeria investir tempo):

1. **Tunar hiperparâmetro do Random Forest e do Extra Trees especificamente** — os parâmetros usados
   aqui (`n_estimators`, `max_depth`, `min_samples_leaf`) nunca foram formalmente tunados (o notebook
   `11` tunou CatBoost/XGBoost/LightGBM, que no fim não fizeram parte do modelo vencedor).
2. **Investigar por que boosting generaliza pior aqui** — se isso for confirmado com mais rigor
   (ex.: comparando curvas de calibração entre train/Kaggle), pode virar um argumento forte para a
   PRT Seguros sobre como montar a próxima base de treino/teste de forma mais consistente.
3. Ver `notebooks/modelagem_caue/experimentos_descartados/` para o histórico completo do que foi
   tentado e não funcionou — evita repetir caminhos já testados.
